In [0]:
filepath ="/Volumes/workspace/raw/a/1383534.json"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lit

# Initialize Spark Session
spark = SparkSession.builder.appName("Cricket").getOrCreate()

# Load the JSON file
# For an uploaded file, you'd typically read it from a path or
# convert the content to an RDD/DataFrame.
# Assuming '1383534.json' is accessible in the Spark environment
df = spark.read.option("multiLine", True).json(filepath)

# --- 1. Flattening Match Info ---
match_info_df = df.select(
    col("meta.data_version").alias("data_version"),
    col("meta.created").alias("created_date"),
    col("meta.revision").alias("revision"),
    col("info.balls_per_over").alias("balls_per_over"),
    col("info.city").alias("city"),
    explode(col("info.dates")).alias("match_date"), # Dates is an array, so explode
    col("info.event.name").alias("event_name"),
    col("info.event.group").alias("event_group"),
    col("info.gender").alias("gender"),
    col("info.match_type").alias("match_type"),
    col("info.outcome.winner").alias("outcome_winner"),
    col("info.outcome.by.wickets").alias("outcome_by_wickets"),
    col("info.overs").alias("total_overs"),
    col("info.season").alias("season"),
    col("info.team_type").alias("team_type"),
    col("info.toss.decision").alias("toss_decision"),
    col("info.toss.winner").alias("toss_winner"),
    col("info.venue").alias("venue")
)
# Add a unique match_id if needed, for joining later
match_info_df = match_info_df.withColumn("match_id", lit("match_1383534")) # Example ID

match_info_df.createOrReplaceTempView("match_info")


# --- 2. Flattening Teams ---
# Teams are in an array under info.teams
teams_df = df.select(explode(col("info.teams")).alias("team_name"))
teams_df = teams_df.withColumn("match_id", lit("match_1383534"))
teams_df.createOrReplaceTempView("teams")


# --- 3. Flattening Players ---
# Players are nested under info.players by team name
# This requires iterating through the player structure
players_data = []
for team_name in df.select(explode(col("info.teams"))).collect():
    team = team_name["col"]
    team_players = df.select(col(f"info.players.`{team}`").alias("players_list")).first()
    if team_players and team_players["players_list"]:
        for player in team_players["players_list"]:
            player_id = df.select(col(f"info.registry.people.`{player}`")).first()[0]
            players_data.append(("match_1383534", team, player, player_id))

players_df = spark.createDataFrame(players_data, ["match_id", "team_name", "player_name", "player_id"])
players_df.createOrReplaceTempView("players")


# --- 4. Flattening Officials ---
# Match referees and umpires are separate arrays
match_referees_df = df.select(explode(col("info.officials.match_referees")).alias("official_name")) \
                      .withColumn("official_type", lit("match_referee"))
match_referees_df = match_referees_df.withColumn("match_id", lit("match_1383534"))

umpires_df = df.select(explode(col("info.officials.umpires")).alias("official_name")) \
                .withColumn("official_type", lit("umpire"))
umpires_df = umpires_df.withColumn("match_id", lit("match_1383534"))

officials_df = match_referees_df.unionAll(umpires_df)
officials_df.createOrReplaceTempView("officials")


# --- 5. Flattening Innings and Deliveries ---
# This is the most complex part, involving nested arrays.
# We'll first explode innings, then deliveries within each inning.

innings_exploded_df = df.select(explode(col("innings")).alias("inning"))
innings_df = innings_exploded_df.select(
    col("inning.team").alias("inning_team"),
    col("inning.target.overs").alias("inning_target_overs"),
    col("inning.target.runs").alias("inning_target_runs")
)
innings_df = innings_df.withColumn("match_id", lit("match_1383534"))
innings_df.createOrReplaceTempView("innings")

# For Deliveries (nested within innings)
deliveries_df = innings_exploded_df.select(
    col("inning.team").alias("inning_team"),
    explode(col("inning.overs")).alias("over_data")
).select(
    col("inning_team"),
    col("over_data.over").alias("over_number"),
    explode(col("over_data.deliveries")).alias("delivery_data")
).select(
    col("inning_team"),
    col("over_number"),
    col("delivery_data.batter").alias("batter"),
    col("delivery_data.bowler").alias("bowler"),
    col("delivery_data.non_striker").alias("non_striker"),
    col("delivery_data.runs.batter").alias("runs_batter"),
    col("delivery_data.runs.extras").alias("runs_extras"),
    col("delivery_data.runs.total").alias("runs_total"),
    col("delivery_data.extras.legbyes").alias("extras_legbyes"),
    col("delivery_data.extras.wides").alias("extras_wides"),
    col("delivery_data.extras.byes").alias("extras_byes"),
    col("delivery_data.wickets").alias("wickets_array") # Keep as array for next step
)
deliveries_df = deliveries_df.withColumn("match_id", lit("match_1383534"))
deliveries_df.createOrReplaceTempView("deliveries")


# --- 6. Flattening Wickets (from deliveries) ---
# Wickets are within the deliveries array, so we need to explode that as well
wickets_df = deliveries_df.filter(col("wickets_array").isNotNull()) \
                          .select(
                              col("match_id"),
                              col("inning_team"),
                              col("over_number"),
                              col("batter"), # The batter at the time of wicket
                              col("bowler"), # The bowler at the time of wicket
                              explode(col("wickets_array")).alias("wicket_details")
                          ).select(
                              col("match_id"),
                              col("inning_team"),
                              col("over_number"),
                              col("batter"),
                              col("bowler"),
                              col("wicket_details.player_out").alias("player_out"),
                              col("wicket_details.kind").alias("wicket_kind"),
                              explode(col("wicket_details.fielders")).alias("fielder_details") # fielders is an array
                          ).select(
                              col("match_id"),
                              col("inning_team"),
                              col("over_number"),
                              col("batter"),
                              col("bowler"),
                              col("player_out"),
                              col("wicket_kind"),
                              col("fielder_details.name").alias("fielder_name")
                          )
wickets_df.createOrReplaceTempView("wickets")


# --- 7. Flattening Powerplays ---
# Powerplays are nested within each inning
powerplays_df = innings_exploded_df.select(
    col("inning.team").alias("inning_team"),
    explode(col("inning.powerplays")).alias("powerplay_details")
).select(
    col("inning_team"),
    col("powerplay_details.from").alias("powerplay_from"),
    col("powerplay_details.to").alias("powerplay_to"),
    col("powerplay_details.type").alias("powerplay_type")
)
powerplays_df = powerplays_df.withColumn("match_id", lit("match_1383534"))
powerplays_df.createOrReplaceTempView("powerplays")


# Show some examples of the flattened tables
print("Match Info:")
spark.sql("SELECT * FROM match_info").show()

print("\nTeams:")
spark.sql("SELECT * FROM teams").show()

print("\nPlayers:")
spark.sql("SELECT * FROM players").show()

print("\nOfficials:")
spark.sql("SELECT * FROM officials").show()

print("\nInnings:")
spark.sql("SELECT * FROM innings").show()

print("\nDeliveries (first few rows and selected columns):")
spark.sql("SELECT match_id, inning_team, over_number, batter, bowler, runs_total FROM deliveries LIMIT 10").show()

print("\nWickets:")
spark.sql("SELECT * FROM wickets").show()

print("\nPowerplays:")
spark.sql("SELECT * FROM powerplays").show()

# Stop Spark Session
# spark.stop()

In [0]:
display(deliveries_df)

In [0]:
spark.sql("SELECT * FROM deliveries").show()